In [32]:
import pandas as pd
import numpy as np

In [33]:
from google.colab import files
upload = files.upload()

Saving final_synthetic_dataset.csv to final_synthetic_dataset (2).csv


In [34]:
df = pd.read_csv("final_synthetic_dataset.csv")

In [35]:
df["aadhaar_sim"] = (df["user_aadhaar"] == df["record_aadhaar"]).astype(int)

In [36]:
df.info()
df.describe()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   user_name       5000 non-null   object 
 1   record_name     5000 non-null   object 
 2   user_dob        5000 non-null   object 
 3   record_dob      5000 non-null   object 
 4   user_address    5000 non-null   object 
 5   record_address  5000 non-null   object 
 6   user_aadhaar    5000 non-null   int64  
 7   record_aadhaar  5000 non-null   int64  
 8   user_voter      3470 non-null   object 
 9   record_voter    3302 non-null   object 
 10  name_sim        5000 non-null   float64
 11  dob_sim         5000 non-null   int64  
 12  addr_sim        5000 non-null   float64
 13  id_sim          5000 non-null   float64
 14  conflict_score  5000 non-null   float64
 15  label           5000 non-null   object 
 16  scenario        5000 non-null   object 
 17  aadhaar_sim     5000 non-null   i

,user_name,record_name,user_dob,record_dob,user_address,record_address,user_aadhaar,record_aadhaar,user_voter,record_voter,name_sim,dob_sim,addr_sim,id_sim,conflict_score,label,scenario,aadhaar_sim
0,Arjun Agarwal,Arjun Agarwal,2002-04-01,2002-04-01,"409/6, Street-23B, Sector-20, Opp Market, Noid...","409/6, Street-23B, Sector-20, Opp Market, Noid...",837288924171,837288924171,NaN,NaN,1.000000,1,1.000000,1.000000,-4.440892e-17,Verified,perfect,1
1,Pooja Singh,pooja singh,1981-02-24,1981-02-24,"944/4, Street-43B, Sector-9, Near Temple, Noid...","944/4, Street-43B, Sector-9, Near Temple, Noid...",345208327770,345208327770,NaN,NaN,0.878788,1,1.000000,1.000000,3.636364e-02,Verified,name_case,1
2,Ankit Kumar,Ankit Kumar,1963-08-27,1963-08-27,"393/1, Street-43B, Sector-7, Opp Market, Delhi...","393/1, Street-43B, Sector-7, Opp Market, Delhi...",34388949197,34388949197,OSE5829521,OSE5829521,1.000000,1,0.779915,1.000000,4.401692e-02,Verified,address_pin_change,1
3,Neha Singh,Neha Singh,1980-07-27,1980-07-27,"324/19, Street-19A, Sector-40, Near Temple, Lu...","324/19, Street-19A, Sector-40, Near Temple, Lu...",142444771153,142444771153,OUU0382937,OUU0382937,1.000000,1,1.000000,1.000000,-1.332268e-16,Verified,perfect,1
4,Karan Singh,Karan Singh,1987-02-02,1987-02-02,"531/15, Street-32A, Sector-38, Opp Market, Noi...","531/15, Street-32A, Sector-38, Opp Market, Noi...",744100434740,744100454740,NaN,NaN,1.000000,1,1.000000,0.916667,2.083333e-02,Verified,aadhaar_partial,0


In [37]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 5000
Number of columns: 18


In [38]:
df.isnull().sum()

,0
user_name,0
record_name,0
user_dob,0
record_dob,0
user_address,0
record_address,0
user_aadhaar,0
record_aadhaar,0
user_voter,1530
record_voter,1698


In [39]:
import numpy as np

# push some values toward overlap zones
df.loc[df["label"] == "Verified", "name_sim"] -= np.random.uniform(0, 0.2, len(df[df["label"]=="Verified"]))
df.loc[df["label"] == "Suspicious", "name_sim"] += np.random.uniform(0, 0.2, len(df[df["label"]=="Suspicious"]))

df["name_sim"] = df["name_sim"].clip(0, 1)

In [40]:
import random

def flip_label(label):
    if random.random() < 0.15:
        return random.choice(["Verified","Review","Suspicious"])
    return label

df["label"] = df["label"].apply(flip_label)

In [41]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,     # more trees
    max_depth=9,          # slightly deeper
    min_samples_split=5,  # prevents overfitting
    random_state=42
)

In [42]:
X = df[[
    "name_sim",
    "dob_sim",
    "addr_sim",
    "aadhaar_sim",
    "id_sim"
]]
y = df["label"]

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [44]:
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",9
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [45]:
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.922
[[ 79   2  36]
 [  2  94  32]
 [  4   2 749]]


TESTING

In [46]:
test_1 = {
    "name_sim": 1.0,
    "dob_sim": 1,
    "addr_sim": 1.0,
    "aadhaar_sim": 1,
    "id_sim": 1
}
test_2 = {
    "name_sim": 0.92,   # small typo
    "dob_sim": 1,
    "addr_sim": 0.90,   # Rd vs Road
    "aadhaar_sim": 1,
    "id_sim": 1
}
test_3 = {
    "name_sim": 1.0,
    "dob_sim": 1,
    "addr_sim": 1.0,
    "aadhaar_sim": 0,   # mismatch
    "id_sim": 1
}
test_4 = {
    "name_sim": 1.0,
    "dob_sim": 0,
    "addr_sim": 1.0,
    "aadhaar_sim": 1,
    "id_sim": 1
}

In [47]:
import pandas as pd

tests = [test_1, test_2, test_3, test_4]

for i, test in enumerate(tests):
    df_test = pd.DataFrame([test])

    ml_pred = model.predict(df_test)[0]

    # apply your rules
    if test["dob_sim"] == 0 or test["aadhaar_sim"] == 0 or test["id_sim"] == 0:
        final = "Suspicious"
    else:
        final = ml_pred

    print(f"Test {i+1}: {final}")

Test 1: Verified
Test 2: Verified
Test 3: Suspicious
Test 4: Suspicious


In [48]:
import pandas as pd

def final_decision(input_data, ml_prediction):

    # HARD RULES
    if input_data["dob_sim"] == 0:
        return "Suspicious"
    if input_data["aadhaar_sim"] == 0:
        return "Suspicious"
    if input_data["id_sim"] == 0:
        return "Suspicious"

    # FORCE REVIEW
    if input_data["name_sim"] <= 0.90 or input_data["addr_sim"] <= 0.90:
        return "Review"

    return "Verified"


test_review = {
    "name_sim": 0.85,
    "dob_sim": 1,
    "addr_sim": 0.80,
    "aadhaar_sim": 1,
    "id_sim": 1
}

df_test = pd.DataFrame([test_review])

ml_pred = model.predict(df_test)[0]

final = final_decision(test_review, ml_pred)

print("ML Prediction:", ml_pred)
print("Final Output:", final)

ML Prediction: Verified
Final Output: Review


In [49]:
import pickle

# save model
with open("voter_rf_model.pkl", "wb") as f:
    pickle.dump(model, f)

In [50]:
from google.colab import files
files.download("voter_rf_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>